In [1]:
from pathlib import Path
import pandas as pd
import re

from parse_dftsp_grepscf_to_df import parse_grepscf
from universal import *

from time import asctime

## def function: Combine energy calculation
combine energy per TiO2 unit $$E_{comb} = E_{n}/n - E_1$$

In [2]:
def get_combine_energy(df: pd.DataFrame) -> pd.DataFrame:
    # format the output df
    column_comb = [x.split(" ")[0]+" eV" for x in df.columns]
    index_comb = df.index
    df_comb = pd.DataFrame(columns=column_comb, index=index_comb)

    # calculate the values
    for i in [x.split(" ")[0] for x in df.columns]:
        n = re.search(r'^(\d+)[a-z]+', i).group(1)
        n = int(n)
        # print(n)
        E_i_name = i + " au"
        dE_i_name = i + " eV"
        df_comb.loc[:,dE_i_name] = \
            (df.loc[:, E_i_name] / n - df.loc[:, "1a au"]) * hartree_to_eV

    return df_comb



# Optimization methods own energies

### MLIP energies

In [28]:
mlE_dir = Path("tio2_ChenDixon_out")
modals = [x for x in mlE_dir.iterdir() if x.is_dir()]
mlE_path = [x / "energies.csv" for x in modals]

# print(modals)
# print(mlE_path)

df_ml = [pd.read_csv(x) for x in mlE_path]



# sort file values like: "40a.xyz", "5b.xyz", "64b.xyz" by number then name
def sort_rows(df: pd.DataFrame, col_name: str = "file"):
    if col_name == "index":
        tmp = df.index.str.extract(r"^(\d+)([a-z])", expand=True)
        tmp.loc[:,"ind"] = df.index
        tmp.set_index("ind", inplace=True)
        tmp.loc[:,0] = tmp.loc[:,0].astype(int)
        df = pd.concat([df, tmp], axis=1)
        df_sorted = (
            df.sort_values([0, 1])
              .drop(columns=[0, 1])
        )
    else:
        tmp = df[col_name].str.extract(r"^(\d+)([a-z])", expand=True)  # number, letter
        df_sorted = (
            df.assign(_n=tmp[0].astype(int), _s=tmp[1].str.lower())
              .sort_values(["_n", "_s"])
              .drop(columns=["_n", "_s"])
              .reset_index(drop=True)
        )
    return df_sorted

# substract .xyz in file column
def rename_sub_xyz(df: pd.DataFrame):
    df.loc[:,"file"] = df.loc[:, "file"].str.split(".").str[0]
    return df


df_ml_sorted = []
for i, df in enumerate(df_ml):
    # sort and reformat file column
    df_sorted = sort_rows(df)
    df_sorted = rename_sub_xyz(df_sorted)

    # only keep E_final and rename it to the modal name
    df_sorted = df_sorted.drop(columns=["E_initial", "dE"])
    modal_name = modals[i].stem
    df_sorted = df_sorted.rename(columns={"E_final": modal_name})

    # rename file column to Method, and set it as index
    df_sorted.rename(columns={"file": "Method"}, inplace=True)
    df_sorted.set_index("Method", inplace=True)

    df_ml_sorted.append(df_sorted)


print(len(df_ml_sorted))
df_ml_tot = pd.concat(df_ml_sorted, axis=1)
df_ml_tot = sort_rows(df_ml_tot, col_name="index")
df_ml_tot = df_ml_tot.T

# print(df_ml_tot.columns.values)
df_ml_au = df_ml_tot.copy()
df_ml_au.columns = [x+" au" for x in df_ml_tot.columns]
df_ml_au = df_ml_au / hartree_to_eV

df_ml_au

5


,1a au,5a au,5b au,8a au,8b au,12a au,12b au,12c au,12d au,40a au,40b au,40c au,64a au,64b au
matpes_pbe,-0.776273,-4.599709,-4.598634,-7.518393,-7.513435,NaN,NaN,NaN,NaN,-38.634489,-38.634260,-38.495221,-61.884436,-61.934958
omat24,-0.776108,-4.600575,-4.601135,-7.520506,-7.516646,NaN,NaN,NaN,NaN,-38.636808,-38.637167,-38.504166,-61.883813,-61.935370
omol25_low,-999.914524,-5000.362586,-5000.369477,-8000.775828,-8000.778125,-12001.322224,-12001.330263,-12001.306146,-12001.136180,-40005.179147,-40005.050525,-40005.105649,-64008.531019,-64008.462114
mpa,-0.776350,-4.611786,-4.610676,-7.534578,-7.531578,NaN,NaN,NaN,NaN,-38.718439,-38.691515,-38.587718,-62.032663,-62.073885
matpes_r2scan,-1.035827,-5.973159,-5.975542,-9.742232,-9.738324,-14.761718,-14.757045,-14.736577,-14.587926,-49.964051,-49.881584,-49.861725,-80.131725,-80.106021


### DFT opt energies

In [33]:
df_dft = pd.read_csv("dft_energies_tot_opt.csv")

# combine Method and Basis, and delete 3rd, 4th columns
df_dft.loc[:,"Method"] = df_dft.loc[:,"Method"] + '_' + df_dft.loc[:,"Basis"]
df_dft.drop(columns=["Basis"], inplace=True)

# copy pbe0svp optimized pbe0tzvp method to pbe0tzvp 1
df_dft.iloc[2, 8:12] = df_dft.iloc[3, 8:12]
df_dft.drop([3], inplace=True)

df_dft.drop(columns=["Dispersion","original0/optimized1"], inplace=True)
df_dft.set_index("Method", inplace=True)

df_dft

,1a au,5a au,5b au,8a au,8b au,12a au,12b au,12c au,12d au,40a au,40b au,40c au,64a au,64b au
Method,,,,,,,,,,,,,,
article_article,-999.890386,-5000.191792,-5000.191502,-8000.481795,-8000.477682,-12000.856365,-12000.815155,-12000.799997,-12000.720337,-40003.545213,-40003.536524,-40003.423463,-64005.820291,-64005.817521
pbe0_def2svp,-999.357624,-4997.617272,-4997.628051,-7996.396703,-7996.401048,-11994.753940,-11994.730495,-11994.702984,-11994.578923,NaN,NaN,NaN,NaN,NaN
pbe0_def2tzvp,-999.652860,-4999.060280,-4999.064222,-7998.690832,-7998.690928,-11998.187445,-11998.161345,-11998.141559,-11998.011492,NaN,NaN,NaN,NaN,NaN
pbe_def2svp,-999.380991,-4997.691722,-4997.699375,-7996.501134,-7996.504978,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
pbe_def2tzvp,-999.678333,-4999.131012,-4999.133300,-7998.788779,-7998.788668,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### combine two df and calculate df_comb

In [34]:
# concat two df
df_ownE = pd.concat([df_dft, df_ml_au], axis=0)

df_ownE

,1a au,5a au,5b au,8a au,8b au,12a au,12b au,12c au,12d au,40a au,40b au,40c au,64a au,64b au
article_article,-999.890386,-5000.191792,-5000.191502,-8000.481795,-8000.477682,-12000.856365,-12000.815155,-12000.799997,-12000.720337,-40003.545213,-40003.536524,-40003.423463,-64005.820291,-64005.817521
pbe0_def2svp,-999.357624,-4997.617272,-4997.628051,-7996.396703,-7996.401048,-11994.753940,-11994.730495,-11994.702984,-11994.578923,NaN,NaN,NaN,NaN,NaN
pbe0_def2tzvp,-999.652860,-4999.060280,-4999.064222,-7998.690832,-7998.690928,-11998.187445,-11998.161345,-11998.141559,-11998.011492,NaN,NaN,NaN,NaN,NaN
pbe_def2svp,-999.380991,-4997.691722,-4997.699375,-7996.501134,-7996.504978,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
pbe_def2tzvp,-999.678333,-4999.131012,-4999.133300,-7998.788779,-7998.788668,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
matpes_pbe,-0.776273,-4.599709,-4.598634,-7.518393,-7.513435,NaN,NaN,NaN,NaN,-38.634489,-38.634260,-38.495221,-61.884436,-61.934958
omat24,-0.776108,-4.600575,-4.601135,-7.520506,-7.516646,NaN,NaN,NaN,NaN,-38.636808,-38.637167,-38.504166,-61.883813,-61.935370
omol25_low,-999.914524,-5000.362586,-5000.369477,-8000.775828,-8000.778125,-12001.322224,-12001.330263,-12001.306146,-12001.136180,-40005.179147,-40005.050525,-40005.105649,-64008.531019,-64008.462114
mpa,-0.776350,-4.611786,-4.610676,-7.534578,-7.531578,NaN,NaN,NaN,NaN,-38.718439,-38.691515,-38.587718,-62.032663,-62.073885
matpes_r2scan,-1.035827,-5.973159,-5.975542,-9.742232,-9.738324,-14.761718,-14.757045,-14.736577,-14.587926,-49.964051,-49.881584,-49.861725,-80.131725,-80.106021


In [35]:
df_ownE_comb = get_combine_energy(df_ownE)

print(df_ownE_comb.index.tolist())
df_ownE_comb

['article_article', 'pbe0_def2svp', 'pbe0_def2tzvp', 'pbe_def2svp', 'pbe_def2tzvp', 'matpes_pbe', 'omat24', 'omol25_low', 'mpa', 'matpes_r2scan']


,1a eV,5a eV,5b eV,8a eV,8b eV,12a eV,12b eV,12c eV,12d eV,40a eV,40b eV,40c eV,64a eV,64b eV
article_article,0.0,-4.026537,-4.024959,-4.621541,-4.607551,-4.924659,-4.831211,-4.796838,-4.6162,-5.394507,-5.388596,-5.311682,-5.457412,-5.456235
pbe0_def2svp,0.0,-4.51249,-4.571155,-5.223622,-5.238401,-5.583912,-5.530748,-5.468363,-5.187039,NaN,NaN,NaN,NaN,NaN
pbe0_def2tzvp,0.0,-4.331948,-4.353402,-4.993134,-4.993459,-5.33599,-5.276805,-5.231937,-4.936995,NaN,NaN,NaN,NaN,NaN
pbe_def2svp,0.0,-4.281818,-4.323468,-4.942983,-4.956061,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
pbe_def2tzvp,0.0,-4.02374,-4.036194,-4.633138,-4.632761,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
matpes_pbe,0.0,-3.90944,-3.903593,-4.449787,-4.432922,NaN,NaN,NaN,NaN,-5.159,-5.158845,-5.064259,-5.188446,-5.209927
omat24,0.0,-3.918639,-3.921683,-4.461456,-4.448328,NaN,NaN,NaN,NaN,-5.16506,-5.165305,-5.074826,-5.192663,-5.214584
omol25_low,0.0,-4.299219,-4.336719,-4.964844,-4.972656,-5.324219,-5.342448,-5.28776,-4.902344,-5.849219,-5.761719,-5.799219,-5.953125,-5.923828
mpa,0.0,-3.973055,-3.967011,-4.502728,-4.492523,NaN,NaN,NaN,NaN,-5.213998,-5.195681,-5.125069,-5.249355,-5.266882
matpes_r2scan,0.0,-4.321299,-4.334269,-4.95117,-4.937876,-5.287616,-5.277019,-5.230607,-4.893522,-5.803492,-5.747392,-5.733882,-5.883955,-5.873026


# tzvp SP energies

### read all sp energies from grepscf to df

In [7]:
# read grepscf txt files
folder = Path("tio2_ChenDixon_gjf")
path_1 = folder / "all1_sp" / "energies.txt"
path_5 = folder / "all5_sp" / "energies.txt"
path_8 = folder / "all8_sp" / "energies.txt"

df_1 = parse_grepscf(path_1)
df_5 = parse_grepscf(path_5)
df_8 = parse_grepscf(path_8)

df_list = [df_1, df_5, df_8]

In [8]:
df_8

,structure,opt method,sp method,energy_au
0,8a,article_article,pbe0_def2tzvp,-7998.680535
1,8a,matpes_pbe,pbe0_def2tzvp,-7998.675239
2,8a,matpes_r2scan,pbe0_def2tzvp,-7998.684010
3,8a,mpa_mpa,pbe0_def2tzvp,-7998.675920
4,8a,omat24_omat24,pbe0_def2tzvp,-7998.675645
5,8a,omol25_low,pbe0_def2tzvp,-7998.689734
6,8a,pbe0_def2svp,pbe0_def2tzvp,-7998.689023
7,8a,pbe_def2svp,pbe0_def2tzvp,-7998.688561
8,8a,article_article,pbe_def2tzvp,-7998.787842
9,8a,matpes_pbe,pbe_def2tzvp,-7998.786960


### merge to two groups: df_pbe0, df_pbe

In [9]:
# split to two groups: pbe0, pbe
df_list_pbe0 = [df[df["sp method"] == "pbe0_def2tzvp"] for df in df_list]
df_list_pbe = [df[df["sp method"] == "pbe_def2tzvp"] for df in df_list]


# matching and sorting to methods list
def match_and_sort(df: pd.DataFrame) -> pd.DataFrame:
    """
    Extract rows where 'opt method' matches methods list,
    sorted by methods order.
    """
    methods = ['article_article', 'pbe0_def2svp', 'pbe0_def2tzvp', 'pbe_def2svp', 'pbe_def2tzvp', 'matpes_pbe', 'omat24_omat24', 'omol25_low', 'mpa_mpa', 'matpes_r2scan']

    # Filter rows matching methods
    mask = df['opt method'].isin(methods)
    filtered = df[mask].copy()

    # Sort by methods order (stable sort)
    filtered['sort_key'] = filtered['opt method'].map({m: i for i, m in enumerate(methods)})
    result = filtered.sort_values('sort_key').drop(columns='sort_key')

    return result


df_list_pbe0 = [match_and_sort(df) for df in df_list_pbe0]
df_list_pbe = [match_and_sort(df) for df in df_list_pbe]

df_list_pbe0 = [df.drop(columns=["sp method"]) for df in df_list_pbe0]
df_list_pbe = [df.drop(columns=["sp method"]) for df in df_list_pbe]

for i in df_list_pbe0+df_list_pbe:
    print(len(i.index))

# print(df_list_pbe0[2])


10
20
16
10
20
16


In [10]:
df_list_pbe0[2].reset_index(drop=True)

,structure,opt method,energy_au
0,8a,article_article,-7998.680535
1,8b,article_article,-7998.680393
2,8b,pbe0_def2svp,-7998.689135
3,8a,pbe0_def2svp,-7998.689023
4,8a,pbe_def2svp,-7998.688561
5,8b,pbe_def2svp,-7998.688511
6,8b,matpes_pbe,-7998.674781
7,8a,matpes_pbe,-7998.675239
8,8b,omat24_omat24,-7998.675534
9,8a,omat24_omat24,-7998.675645


In [11]:
def reformat_dftsp_df(df: pd.DataFrame) -> pd.DataFrame:
    """separate a and b"""
    df_out = pd.DataFrame(index=['article_article', 'pbe0_def2svp', 'pbe0_def2tzvp', 'pbe_def2svp', 'pbe_def2tzvp', 'matpes_pbe', 'omat24_omat24', 'omol25_low', 'mpa_mpa', 'matpes_r2scan'])
    df = df.reset_index(drop=True)

    for i in range(len(df.index)):
        structure = df.loc[i, "structure"]
        energy = df.loc[i, "energy_au"]

        new_column_name = structure + " au"
        new_index = df.loc[i, "opt method"]

        df_out.loc[new_index, new_column_name] = energy

    return df_out

reformat_dftsp_df(df_list_pbe0[2])

,8a au,8b au
article_article,-7998.680535,-7998.680393
pbe0_def2svp,-7998.689023,-7998.689135
pbe0_def2tzvp,NaN,NaN
pbe_def2svp,-7998.688561,-7998.688511
pbe_def2tzvp,NaN,NaN
matpes_pbe,-7998.675239,-7998.674781
omat24_omat24,-7998.675645,-7998.675534
omol25_low,-7998.689734,-7998.689868
mpa_mpa,-7998.675920,-7998.675791
matpes_r2scan,-7998.684010,-7998.683693


In [12]:
df_list_pbe0_reformat = [reformat_dftsp_df(df) for df in df_list_pbe0]
df_list_pbe_reformat = [reformat_dftsp_df(df) for df in df_list_pbe]

df_list_pbe0_reformat[1]

,5a au,5b au
article_article,-4999.054146,-4999.057686
pbe0_def2svp,-4999.058998,-4999.063063
pbe0_def2tzvp,-4999.060280,-4999.064222
pbe_def2svp,-4999.058532,-4999.062800
pbe_def2tzvp,-4999.055563,-4999.059600
matpes_pbe,-4999.050827,-4999.054039
omat24_omat24,-4999.051386,-4999.054621
omol25_low,-4999.059019,-4999.063340
mpa_mpa,-4999.051394,-4999.054309
matpes_r2scan,-4999.055888,-4999.059767


In [13]:
df_pbe_tot = pd.concat(df_list_pbe_reformat, axis=1)
df_pbe0_tot = pd.concat(df_list_pbe0_reformat, axis=1)

df_pbe0_tot

,1a au,5a au,5b au,8a au,8b au
article_article,-999.651649,-4999.054146,-4999.057686,-7998.680535,-7998.680393
pbe0_def2svp,-999.652513,-4999.058998,-4999.063063,-7998.689023,-7998.689135
pbe0_def2tzvp,-999.652860,-4999.060280,-4999.064222,NaN,NaN
pbe_def2svp,-999.652330,-4999.058532,-4999.062800,-7998.688561,-7998.688511
pbe_def2tzvp,-999.651774,-4999.055563,-4999.059600,NaN,NaN
matpes_pbe,-999.650344,-4999.050827,-4999.054039,-7998.675239,-7998.674781
omat24_omat24,-999.650574,-4999.051386,-4999.054621,-7998.675645,-7998.675534
omol25_low,-999.652064,-4999.059019,-4999.063340,-7998.689734,-7998.689868
mpa_mpa,-999.650206,-4999.051394,-4999.054309,-7998.675920,-7998.675791
matpes_r2scan,-999.651591,-4999.055888,-4999.059767,-7998.684010,-7998.683693


In [14]:
df_pbe_tot

,1a au,5a au,5b au,8a au,8b au
article_article,-999.678306,-4999.130072,-4999.132496,-7998.787842,-7998.787465
pbe0_def2svp,-999.676157,-4999.121563,-4999.123890,-7998.774426,-7998.773949
pbe0_def2tzvp,-999.677308,-4999.126568,-4999.128925,NaN,NaN
pbe_def2svp,-999.677931,-4999.129887,-4999.132172,-7998.787115,-7998.786961
pbe_def2tzvp,-999.678333,-4999.131012,-4999.133300,NaN,NaN
matpes_pbe,-999.677976,-4999.129773,-4999.131934,-7998.786960,-7998.786760
omat24_omat24,-999.678091,-4999.130033,-4999.132106,-7998.787062,-7998.786959
omol25_low,-999.677023,-4999.127026,-4999.130478,-7998.784245,-7998.784818
mpa_mpa,-999.677985,-4999.130069,-4999.131994,-7998.787075,-7998.787039
matpes_r2scan,-999.678178,-4999.129994,-4999.132919,-7998.788403,-7998.788164


### get comb energy df

In [15]:
df_pbe0_comb = get_combine_energy(df_pbe0_tot)
df_pbe_comb = get_combine_energy(df_pbe_tot)

df_pbe0_comb

,1a eV,5a eV,5b eV,8a eV,8b eV
article_article,0.0,-4.331521,-4.350786,-4.991061,-4.990577
pbe0_def2svp,0.0,-4.334427,-4.35655,-4.996431,-4.996815
pbe0_def2tzvp,0.0,-4.331948,-4.353402,NaN,NaN
pbe_def2svp,0.0,-4.336845,-4.360071,-4.999814,-4.999644
pbe_def2tzvp,0.0,-4.335818,-4.357789,NaN,NaN
matpes_pbe,0.0,-4.34896,-4.366444,-5.008551,-5.006993
omat24_omat24,0.0,-4.345747,-4.363351,-5.003676,-5.003299
omol25_low,0.0,-4.34674,-4.370254,-5.01105,-5.011504
mpa_mpa,0.0,-4.355793,-4.371656,-5.014615,-5.014177
matpes_r2scan,0.0,-4.342569,-4.363681,-5.004448,-5.003372


# Results

Energies from theirself methods

In [16]:
df_ownE

,1a au,5a au,5b au,8a au,8b au,40a au,40b au,40c au,64a au,64b au
article_article,-999.890386,-5000.191792,-5000.191502,-8000.481795,-8000.477682,-40003.545213,-40003.536524,-40003.423463,-64005.820291,-64005.817521
pbe0_def2svp,-999.357624,-4997.617272,-4997.628051,-7996.396703,-7996.401048,NaN,NaN,NaN,NaN,NaN
pbe0_def2tzvp,-999.652860,-4999.060280,-4999.064222,-7998.690832,-7998.690928,NaN,NaN,NaN,NaN,NaN
pbe_def2svp,-999.380991,-4997.691722,-4997.699375,-7996.501134,-7996.504978,NaN,NaN,NaN,NaN,NaN
pbe_def2tzvp,-999.678333,-4999.131012,-4999.133300,-7998.788779,-7998.788668,NaN,NaN,NaN,NaN,NaN
matpes_pbe,-0.776273,-4.599709,-4.598634,-7.518393,-7.513435,-38.634489,-38.634260,-38.495221,-61.884436,-61.934958
omat24,-0.776108,-4.600575,-4.601135,-7.520506,-7.516646,-38.636808,-38.637167,-38.504166,-61.883813,-61.935370
omol25_low,-999.914524,-5000.362586,-5000.369477,-8000.775828,-8000.778125,-40005.179147,-40005.050525,-40005.105649,-64008.531019,-64008.462114
mpa,-0.776350,-4.611786,-4.610676,-7.534578,-7.531578,-38.718439,-38.691515,-38.587718,-62.032663,-62.073885
matpes_r2scan,-1.035827,-5.973159,-5.975542,-9.742232,-9.738324,-49.964051,-49.881584,-49.861725,-80.131725,-80.106021


In [17]:
df_ownE_comb

,1a eV,5a eV,5b eV,8a eV,8b eV,40a eV,40b eV,40c eV,64a eV,64b eV
article_article,0.0,-4.026537,-4.024959,-4.621541,-4.607551,-5.394507,-5.388596,-5.311682,-5.457412,-5.456235
pbe0_def2svp,0.0,-4.51249,-4.571155,-5.223622,-5.238401,NaN,NaN,NaN,NaN,NaN
pbe0_def2tzvp,0.0,-4.331948,-4.353402,-4.993134,-4.993459,NaN,NaN,NaN,NaN,NaN
pbe_def2svp,0.0,-4.281818,-4.323468,-4.942983,-4.956061,NaN,NaN,NaN,NaN,NaN
pbe_def2tzvp,0.0,-4.02374,-4.036194,-4.633138,-4.632761,NaN,NaN,NaN,NaN,NaN
matpes_pbe,0.0,-3.90944,-3.903593,-4.449787,-4.432922,-5.159,-5.158845,-5.064259,-5.188446,-5.209927
omat24,0.0,-3.918639,-3.921683,-4.461456,-4.448328,-5.16506,-5.165305,-5.074826,-5.192663,-5.214584
omol25_low,0.0,-4.299219,-4.336719,-4.964844,-4.972656,-5.849219,-5.761719,-5.799219,-5.953125,-5.923828
mpa,0.0,-3.973055,-3.967011,-4.502728,-4.492523,-5.213998,-5.195681,-5.125069,-5.249355,-5.266882
matpes_r2scan,0.0,-4.321299,-4.334269,-4.95117,-4.937876,-5.803492,-5.747392,-5.733882,-5.883955,-5.873026


pbe0 tzvp SP energies

In [18]:
df_pbe0_tot

,1a au,5a au,5b au,8a au,8b au
article_article,-999.651649,-4999.054146,-4999.057686,-7998.680535,-7998.680393
pbe0_def2svp,-999.652513,-4999.058998,-4999.063063,-7998.689023,-7998.689135
pbe0_def2tzvp,-999.652860,-4999.060280,-4999.064222,NaN,NaN
pbe_def2svp,-999.652330,-4999.058532,-4999.062800,-7998.688561,-7998.688511
pbe_def2tzvp,-999.651774,-4999.055563,-4999.059600,NaN,NaN
matpes_pbe,-999.650344,-4999.050827,-4999.054039,-7998.675239,-7998.674781
omat24_omat24,-999.650574,-4999.051386,-4999.054621,-7998.675645,-7998.675534
omol25_low,-999.652064,-4999.059019,-4999.063340,-7998.689734,-7998.689868
mpa_mpa,-999.650206,-4999.051394,-4999.054309,-7998.675920,-7998.675791
matpes_r2scan,-999.651591,-4999.055888,-4999.059767,-7998.684010,-7998.683693


In [19]:
df_pbe0_comb

,1a eV,5a eV,5b eV,8a eV,8b eV
article_article,0.0,-4.331521,-4.350786,-4.991061,-4.990577
pbe0_def2svp,0.0,-4.334427,-4.35655,-4.996431,-4.996815
pbe0_def2tzvp,0.0,-4.331948,-4.353402,NaN,NaN
pbe_def2svp,0.0,-4.336845,-4.360071,-4.999814,-4.999644
pbe_def2tzvp,0.0,-4.335818,-4.357789,NaN,NaN
matpes_pbe,0.0,-4.34896,-4.366444,-5.008551,-5.006993
omat24_omat24,0.0,-4.345747,-4.363351,-5.003676,-5.003299
omol25_low,0.0,-4.34674,-4.370254,-5.01105,-5.011504
mpa_mpa,0.0,-4.355793,-4.371656,-5.014615,-5.014177
matpes_r2scan,0.0,-4.342569,-4.363681,-5.004448,-5.003372


pbe tzvp SP energies

In [20]:
df_pbe_tot

,1a au,5a au,5b au,8a au,8b au
article_article,-999.678306,-4999.130072,-4999.132496,-7998.787842,-7998.787465
pbe0_def2svp,-999.676157,-4999.121563,-4999.123890,-7998.774426,-7998.773949
pbe0_def2tzvp,-999.677308,-4999.126568,-4999.128925,NaN,NaN
pbe_def2svp,-999.677931,-4999.129887,-4999.132172,-7998.787115,-7998.786961
pbe_def2tzvp,-999.678333,-4999.131012,-4999.133300,NaN,NaN
matpes_pbe,-999.677976,-4999.129773,-4999.131934,-7998.786960,-7998.786760
omat24_omat24,-999.678091,-4999.130033,-4999.132106,-7998.787062,-7998.786959
omol25_low,-999.677023,-4999.127026,-4999.130478,-7998.784245,-7998.784818
mpa_mpa,-999.677985,-4999.130069,-4999.131994,-7998.787075,-7998.787039
matpes_r2scan,-999.678178,-4999.129994,-4999.132919,-7998.788403,-7998.788164


In [21]:
df_pbe_comb

,1a eV,5a eV,5b eV,8a eV,8b eV
article_article,0.0,-4.019343,-4.032537,-4.63067,-4.629386
pbe0_def2svp,0.0,-4.031531,-4.044196,-4.643532,-4.64191
pbe0_def2tzvp,0.0,-4.027443,-4.040273,NaN,NaN
pbe_def2svp,0.0,-4.028565,-4.040999,-4.638423,-4.637902
pbe_def2tzvp,0.0,-4.02374,-4.036194,NaN,NaN
matpes_pbe,0.0,-4.026718,-4.038475,-4.636669,-4.63599
omat24_omat24,0.0,-4.025003,-4.036287,-4.63389,-4.633541
omol25_low,0.0,-4.03769,-4.056475,-4.653361,-4.655309
mpa_mpa,0.0,-4.028069,-4.038547,-4.636807,-4.636684
matpes_r2scan,0.0,-4.022426,-4.038342,-4.636086,-4.635271
